#  Empirical-Bayes Simulation Notebook

This notebook keeps only the pieces needed to reproduce the final 2x2 figures.

## Usage
1. Set `METHOD = "mom"` in the compute cell and run it.
2. Change to `METHOD = "mle"` and run the same compute cell again.
3. Run the plotting cell to plot whichever of `results_mom` and `results_mle` exist.

The original notebook is left untouched. This cleaned notebook preserves the current simulation logic, the custom `mom` branch, and the optimizer-based `mle` branch used in the existing notebook.


In [ ]:
import time
import numpy as np
from scipy.optimize import minimize
from typing import Literal

# -----------------------
# Choose method here
# Run this cell twice:
# 1) METHOD = "mom"
# 2) METHOD = "mle"
# -----------------------
METHOD = "mle"  # change to "mle" and rerun this cell

# Reproducibility
np.random.seed(7)

# -----------------------
# Global constants
# -----------------------
theta_true_default = 1.0
sigma_e_default = 1.0
sigma_obs_default = 1.0
mu_b_true, sigma_b_true = 0.5, 1.0
sigma_n = 1.0

J_values = np.array([5, 10, 50, 100, 200, 500, 1000])
n_replicates = 2000

sigma_e_grid = [0.5, 1.0, 2.0]
sigma_obs_grid = [0.5, 1.0, 2.0]
theta_grid = [0.0, 1.0, 2.0]

# -----------------------
# Core helper functions
# -----------------------
def simulate_data(J, theta_true, sigma_e, sigma_obs):
    y_e = np.random.normal(theta_true, sigma_e)
    b_j = np.random.normal(mu_b_true, sigma_b_true, size=J)
    y_obs = np.random.normal(theta_true + b_j, sigma_obs, size=J)
    b_null = np.random.normal(mu_b_true, sigma_b_true, size=J)
    y_calib = np.random.normal(b_null, sigma_n, size=J)
    return y_e, y_obs, y_calib

def _estimate_bias(y_calib, method: Literal["mle", "mom"] = "mom"):
    if method == "mle":
        raise ValueError("MLE method not implemented yet.")

    elif method == "mom":
        mu_hat = np.mean(y_calib)
        d = (y_calib - mu_hat)**2
        gamma2 = np.mean(d) - sigma_n**2

        w = 1 / (gamma2 + sigma_n**2)
        w_norm = np.sum(w)
        mu_hat = w * y_calib / w_norm

        return mu_hat, gamma2

def calibrated_mean(y_e, y_obs, mu_b_hat, sig2_b_hat, sigma_e, sigma_obs):
    J = len(y_obs)
    w_e = 1.0 / sigma_e**2
    w_obs = 1.0 / (sigma_obs**2 + sig2_b_hat)
    numer = w_e * y_e + w_obs * np.sum(y_obs - mu_b_hat)
    denom = w_e + J * w_obs
    return numer / denom

# -----------------------
# Optimizer-based MLE branch
# -----------------------
def mle_objective(params, y_rmn, sigma_rmn_2):
    mu_b, sigma_b2 = params
    if sigma_b2 < 0:
        return 1e10

    summands = []
    for j in range(len(y_rmn)):
        num1 = (y_rmn[j] - mu_b) ** 2
        denom1 = sigma_b2 + sigma_rmn_2[j]
        term1 = num1 / (denom1 + 1e-8)
        term2 = np.log(sigma_b2 + sigma_rmn_2[j] + 1e-8)
        summands.append(term1 + term2)

    return np.mean(summands)

def fit_mle_objective(y_rmn, sigma_rmn):
    mu_b_init = np.mean(y_rmn)
    var_sample = np.var(y_rmn, ddof=1)
    avg_noise = np.mean(sigma_rmn**2)
    sigma_b2_init = max(0.0, var_sample - avg_noise)

    init_params = [mu_b_init, sigma_b2_init]
    bounds = [(None, None), (0.0, None)]

    result = minimize(
        fun=mle_objective,
        x0=init_params,
        args=(y_rmn, sigma_rmn**2),
        bounds=bounds,
        method='L-BFGS-B',
        options={"maxiter": 50},
    )

    mu_b_hat, sigma_b2_hat = result.x
    if sigma_b2_hat < 0:
        sigma_b2_hat = 0.0

    return mu_b_hat, sigma_b2_hat

def empirical_bayes_estimator(y_e, y_obs, y_rmn,
                              sigma_e, sigma_obs, sigma_rmn, objective='mle'):
    J = len(y_obs)

    if objective == 'mle':
        mu_b_hat, sigma_b2_hat = fit_mle_objective(y_rmn, sigma_rmn)
    else:
        raise ValueError("Only objective='mle' is supported in this clean notebook.")

    b_hat = []
    for j in range(J):
        denom_j = sigma_rmn[j]**2 + sigma_b2_hat
        w_j = sigma_b2_hat / denom_j
        b_hat_j = w_j * y_rmn[j] + (1.0 - w_j) * mu_b_hat
        b_hat.append(b_hat_j)
    b_hat = np.array(b_hat)

    w_e = 1.0 / (sigma_e**2)
    w_j = 1.0 / (sigma_obs**2)
    numerator = w_e * y_e + w_j * np.sum(y_obs - b_hat)
    denominator = w_e + J * w_j

    return numerator / denominator

# -----------------------
# Simulation driver
# -----------------------
def run_sim(J_vals, theta_true, sigma_e, sigma_obs, estimator=METHOD):
    mse_n, mse_c, ratio = [], [], []
    total_jobs = len(J_vals) * n_replicates
    done = 0
    t0 = time.time()

    for J in J_vals:
        err_n, err_c = [], []
        tJ = time.time()

        for rep in range(n_replicates):
            y_e, y_obs, y_cal = simulate_data(J, theta_true, sigma_e, sigma_obs)
            theta_n = y_e

            if estimator in {"closed_form", "mom"}:
                mu_b, sig2_b = _estimate_bias(y_cal, method=estimator)
                theta_c = calibrated_mean(y_e, y_obs, mu_b, sig2_b, sigma_e, sigma_obs)

            elif estimator == "mle":
                sigma_rmn_vec = np.full(J, sigma_n)
                theta_c = empirical_bayes_estimator(
                    y_e, y_obs, y_cal, sigma_e, sigma_obs, sigma_rmn_vec, objective="mle"
                )

            else:
                raise ValueError("bad estimator")

            err_n.append((theta_n - theta_true) ** 2)
            err_c.append((theta_c - theta_true) ** 2)

            done += 1
            if done % 200 == 0:
                elapsed = time.time() - t0
                rate = done / elapsed
                remain = (total_jobs - done) / rate if rate > 0 else float("inf")
                print(f"{done}/{total_jobs} reps done | elapsed {elapsed:.1f}s | ETA {remain:.1f}s")

        mse_n.append(np.mean(err_n))
        mse_c.append(np.mean(err_c))
        ratio.append(mse_c[-1] / mse_n[-1])
        print(f"Finished J={J} in {time.time() - tJ:.1f}s")

    print(f"Total runtime: {time.time() - t0:.1f}s")
    return np.array(mse_n), np.array(mse_c), np.array(ratio)

# -----------------------
# Save results for the plotting cell
# -----------------------
_result_store = {
    "method": METHOD,
    "J_values": J_values.copy(),
    "n_replicates": n_replicates,
    "sigma_e": {},
    "sigma_obs": {},
    "theta": {},
    "baseline": None,
}

for sig_e in sigma_e_grid:
    mse_n, mse_c, ratio = run_sim(J_values, theta_true_default, sig_e, sigma_obs_default, estimator=METHOD)
    _result_store["sigma_e"][sig_e] = {
        "mse_n": mse_n.copy(),
        "mse_c": mse_c.copy(),
        "ratio": ratio.copy(),
    }

mse_n_ref, mse_c_ref, ratio_ref = run_sim(J_values, theta_true_default, sigma_e_default, sigma_obs_default, estimator=METHOD)
_result_store["baseline"] = {
    "mse_n": mse_n_ref.copy(),
    "mse_c": mse_c_ref.copy(),
    "ratio": ratio_ref.copy(),
}

for sig_o in sigma_obs_grid:
    mse_n, mse_c, ratio = run_sim(J_values, theta_true_default, sigma_e_default, sig_o, estimator=METHOD)
    _result_store["sigma_obs"][sig_o] = {
        "mse_n": mse_n.copy(),
        "mse_c": mse_c.copy(),
        "ratio": ratio.copy(),
    }

for th in theta_grid:
    mse_n, mse_c, ratio = run_sim(J_values, th, sigma_e_default, sigma_obs_default, estimator=METHOD)
    _result_store["theta"][th] = {
        "mse_n": mse_n.copy(),
        "mse_c": mse_c.copy(),
        "ratio": ratio.copy(),
    }

if METHOD == "mom":
    results_mom = _result_store
    print("Saved results to `results_mom`")
elif METHOD == "mle":
    results_mle = _result_store
    print("Saved results to `results_mle`")
else:
    raise ValueError("METHOD must be 'mom' or 'mle'")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": ["Times New Roman", "STIXGeneral", "serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize": 28,
    "axes.labelsize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 14,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

FIGSIZE = (10.5, 4.5)
LW = 2.4
GRID_ALPHA = 0.6

COL_NAIVE = "#6B7280"
COL_RATIO = "#B91C1C"
COL_BETTER = "#16A34A"

COL_SIGE = ["#B45309", "#D97706", "#F59E0B"]
COL_SIGO = ["#166534", "#16A34A", "#4ADE80"]
COL_THETA = ["#5B21B6", "#7C3AED", "#A78BFA"]

sigma_e_grid = [0.5, 1.0, 2.0]
sigma_obs_grid = [0.5, 1.0, 2.0]
theta_grid = [0.0, 1.0, 2.0]

def fit_loglog_slope(x, y):
    lx = np.log10(np.asarray(x, dtype=float))
    ly = np.log10(np.maximum(np.asarray(y, dtype=float), 1e-12))
    slope, intercept = np.polyfit(lx, ly, 1)
    y_fit = 10 ** (intercept + slope * lx)
    return slope, y_fit

def style_axis(ax, xlabel=None, ylabel=None, logy=False):
    ax.set_facecolor("white")
    ax.set_xscale("log")
    if logy:
        ax.set_yscale("log")
        ax.set_ylim(bottom=1e-3)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, which="major", ls=":", alpha=GRID_ALPHA, linewidth=1.0)
    ax.grid(True, which="minor", ls=":", alpha=0.25, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_results_dict(results):
    J_values = np.asarray(results["J_values"])
    method = results["method"]

    fig, axs = plt.subplots(
        2, 2,
        figsize=(FIGSIZE[0] * 1.35, FIGSIZE[1] * 1.9),
        constrained_layout=True,
        facecolor="white"
    )

    slopes_sigma_e = []
    slopes_sigma_obs = []
    slopes_theta = []

    for color, sig_e in zip(COL_SIGE, sigma_e_grid):
        mse_n = np.asarray(results["sigma_e"][sig_e]["mse_n"])
        mse_c = np.asarray(results["sigma_e"][sig_e]["mse_c"])
        slope, y_fit = fit_loglog_slope(J_values, mse_c)
        slopes_sigma_e.append(slope)

        axs[0, 0].plot(J_values, mse_n, ls="--", lw=LW, color=color, alpha=0.28)
        axs[0, 0].plot(
            J_values, mse_c,
            label=rf"$\sigma_e={sig_e}$  |  slope={slope:.2f}",
            marker="o", ms=7, lw=LW, color=color
        )
        axs[0, 0].plot(J_values, y_fit, ls=":", lw=1.6, color=color, alpha=0.9)

    axs[0, 0].set_title(rf"Vary experimental noise $\sigma_e$ ({method})")
    style_axis(axs[0, 0], ylabel="MSE", logy=True)
    axs[0, 0].legend(frameon=False, handlelength=2.4)

    mse_n_ref = np.asarray(results["baseline"]["mse_n"])
    axs[0, 1].plot(J_values, mse_n_ref, ls="--", lw=LW, color=COL_NAIVE, alpha=0.9)

    for color, sig_o in zip(COL_SIGO, sigma_obs_grid):
        mse_c = np.asarray(results["sigma_obs"][sig_o]["mse_c"])
        slope, y_fit = fit_loglog_slope(J_values, mse_c)
        slopes_sigma_obs.append(slope)

        axs[0, 1].plot(
            J_values, mse_c,
            label=rf"$\sigma_o={sig_o}$  |  slope={slope:.2f}",
            marker="s", ms=7, lw=LW, color=color
        )
        axs[0, 1].plot(J_values, y_fit, ls=":", lw=1.6, color=color, alpha=0.9)

    axs[0, 1].set_title(rf"Vary observational noise $\sigma_o$ ({method})")
    style_axis(axs[0, 1], logy=True)
    axs[0, 1].legend(frameon=False, handlelength=2.4)

    axs[1, 0].plot(J_values, mse_n_ref, ls="--", lw=LW, color=COL_NAIVE, alpha=0.9)

    for color, th in zip(COL_THETA, theta_grid):
        mse_c = np.asarray(results["theta"][th]["mse_c"])
        slope, y_fit = fit_loglog_slope(J_values, mse_c)
        slopes_theta.append(slope)

        axs[1, 0].plot(
            J_values, mse_c,
            label=rf"$\theta={th}$  |  slope={slope:.2f}",
            marker="^", ms=7, lw=LW, color=color
        )
        axs[1, 0].plot(J_values, y_fit, ls=":", lw=1.6, color=color, alpha=0.9)

    axs[1, 0].set_title(rf"Vary true effect $\theta$ ({method})")
    style_axis(axs[1, 0], xlabel=r"$J$", ylabel="MSE", logy=True)
    axs[1, 0].legend(frameon=False, handlelength=2.4)

    ratio = np.asarray(results["baseline"]["ratio"])
    axs[1, 1].fill_between(J_values, 0, 1, color=COL_BETTER, alpha=0.08)
    axs[1, 1].axhline(1.0, color=COL_NAIVE, ls="--", lw=1.8, alpha=0.9)
    axs[1, 1].plot(J_values, ratio, marker="o", ms=7, lw=LW, color=COL_RATIO)

    for x, y in zip(J_values, ratio):
        axs[1, 1].annotate(
            f"{y:.2f}",
            xy=(x, y),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=12,
            color=COL_RATIO
        )

    axs[1, 1].set_title(rf"Relative efficiency ({method})")
    style_axis(
        axs[1, 1],
        xlabel=r"$J$",
        ylabel=r"$\mathrm{MSE}_{\mathrm{calib}} / \mathrm{MSE}_{\mathrm{naive}}$"
    )

    plt.show()

    all_slopes = slopes_sigma_e + slopes_sigma_obs + slopes_theta
    print(f"Method: {method}")
    print(f"Average slope over all calibrated curves = {np.mean(all_slopes):.3f}")
    print(f"Average slope for varying sigma_e        = {np.mean(slopes_sigma_e):.3f}")
    print(f"Average slope for varying sigma_o        = {np.mean(slopes_sigma_obs):.3f}")
    print(f"Average slope for varying theta          = {np.mean(slopes_theta):.3f}")
    print()

found_any = False
if "results_mom" in globals():
    plot_results_dict(results_mom)
    found_any = True
if "results_mle" in globals():
    plot_results_dict(results_mle)
    found_any = True
if not found_any:
    raise ValueError("No saved results found. Run the compute cell first.")
